In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load the Dataset

df = pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## Preprocessing the Data
### Drop the unnecessary columns

df.drop(["RowNumber", "CustomerId", "Surname"], axis=1, inplace=True)
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
## Encode the categorical variables
label_encoder = LabelEncoder()
df["Gender"] = label_encoder.fit_transform(df['Gender'])
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [5]:
Geo = df["Geography"].unique()
Geo


<ArrowStringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str

In [6]:
## One-hot encode the 'Geography' column
from sklearn.preprocessing import OneHotEncoder
OHE = OneHotEncoder()
geo_encoded = OHE.fit_transform(df[["Geography"]]).toarray()
geo_encoded

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [7]:
OHE.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [8]:
geo_encoded_df = pd.DataFrame(geo_encoded, columns=OHE.get_feature_names_out())
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [9]:
# Combine the one-hot encoded columns with the original dataframe
df = pd.concat([df, geo_encoded_df], axis=1)    
df.drop("Geography", axis=1, inplace=True)
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [10]:
# Save the encoder and LabelEncoder for future use
with open("label_encoder.pkl", "wb") as le_file:
    pickle.dump(label_encoder, le_file) 

with open("one_hot_encoder.pkl", "wb") as ohe_file:
    pickle.dump(OHE, ohe_file)    

In [11]:
# divide the data into features and target variable
X = df.drop("Exited", axis=1)
y = df["Exited"]    

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [12]:
X_train, X_test

(array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
         -0.57946723, -0.57638802],
        [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
          1.72572313, -0.57638802],
        [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
         -0.57946723,  1.73494238],
        ...,
        [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
         -0.57946723, -0.57638802],
        [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
         -0.57946723, -0.57638802],
        [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
          1.72572313, -0.57638802]]),
 array([[-0.57749609,  0.91324755, -0.6557859 , ..., -0.99850112,
          1.72572313, -0.57638802],
        [-0.29729735,  0.91324755,  0.3900109 , ...,  1.00150113,
         -0.57946723, -0.57638802],
        [-0.52560743, -1.09499335,  0.48508334, ..., -0.99850112,
         -0.57946723,  1.73494238],
        ...,
        [ 0.81311987, -1.09499335,  0.77030065, ...,  

In [13]:
# Save the Scaler for future use
with open("scaler.pkl", "wb") as scaler_file:
    pickle.dump(scaler, scaler_file)        

## ANN Implementation

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense   
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [15]:
X_train.shape[1]

12

In [16]:
## Build the model
ann = Sequential()
ann.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],))) # Hidden layer 1 connected to input layer
ann.add(Dense(32, activation='relu')) # Hidden layer 2
ann.add(Dense(1, activation='sigmoid')) # Output layer

In [17]:
ann.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [18]:
## Compile the model
ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [19]:
## Set the tensorboard callback
log_dir = 'logs/fit/' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [20]:
# Set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [21]:
ann.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32, callbacks=[early_stopping, tensorboard_callback])

Epoch 1/50


250/250 [==============================] - 3s 6ms/step - loss: 0.4504 - accuracy: 0.8108 - val_loss: 0.3896 - val_accuracy: 0.8390
Epoch 2/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3842 - accuracy: 0.8430 - val_loss: 0.3590 - val_accuracy: 0.8560
Epoch 3/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3569 - accuracy: 0.8550 - val_loss: 0.3504 - val_accuracy: 0.8600
Epoch 4/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3471 - accuracy: 0.8537 - val_loss: 0.3467 - val_accuracy: 0.8565
Epoch 5/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3413 - accuracy: 0.8596 - val_loss: 0.3435 - val_accuracy: 0.8580
Epoch 6/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3373 - accuracy: 0.8616 - val_loss: 0.3431 - val_accuracy: 0.8620
Epoch 7/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3348 - accuracy: 0.8624 - val_loss: 0.3388 - val_accuracy: 0.8640
Epoc

In [22]:
ann.save("churn_model.h5")


In [23]:
## Load TensorBoard
%load_ext tensorboard

In [30]:
# %tensorboard --logdir logs/fit/20260505-172025

In [25]:
X_test[97]   

array([-0.59825155, -1.09499335, -1.51143782, -0.3483691 ,  0.45874207,
        0.80843615,  0.64920267, -1.02583358, -0.75935882, -0.99850112,
        1.72572313, -0.57638802])

In [26]:
y_test.iloc[97]

1

In [27]:
## Predicting the results
y_pred = ann.predict([[-0.59825155, -1.09499335, -1.51143782, -0.3483691 ,  0.45874207,
        0.80843615,  0.64920267, -1.02583358, -0.75935882, -0.99850112,
        1.72572313, -0.57638802]])

1/1 [==============================] - 0s 186ms/step


In [28]:
y_pred

array([[0.04908331]], dtype=float32)

In [29]:
if y_pred.squeeze()> 0.5:
    print("The customer is likely to churn.")   
else:
    print("The customer is not likely to churn.")    

The customer is not likely to churn.


In [31]:
print(scaler.feature_names_in_)

['CreditScore' 'Gender' 'Age' 'Tenure' 'Balance' 'NumOfProducts'
 'HasCrCard' 'IsActiveMember' 'EstimatedSalary' 'Geography_France'
 'Geography_Germany' 'Geography_Spain']
